# Notebook 1: Embedding Space Analysis

**Goal:** Understand how the embedding model structures the FMCG corpus.  
**Key questions:**
- Do product categories form natural clusters in embedding space?
- Where do semantically different chunks land near each other (confusion zones)?
- How far apart are chunks from the same document vs. different documents?

**Outputs written to `analysis/outputs/`:**
- `embedding_projections.json` — 2D UMAP coords + metadata (consumed by dashboard)
- `cluster_distances.json` — intra vs inter cluster distance stats

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

from analysis.utils import apply_plot_style, save_output
apply_plot_style()

## 1. Load and Embed the Corpus

In [ ]:
from rag.config_loader import build_embeddings, load_documents, chunk_documents

# Use the baseline config settings
embedding_cfg = {'provider': 'azure_openai', 'model': 'text-embedding-3-small'}
chunking_cfg  = {'strategy': 'fixed', 'chunk_size': 128, 'overlap': 20}

embeddings_model = build_embeddings(embedding_cfg)
docs  = load_documents('data/fmcg_docs')
chunks = chunk_documents(docs, chunking_cfg)

print(f'Documents: {len(docs)}, Chunks: {len(chunks)}')

In [ ]:
texts = [c.page_content for c in chunks]
sources = [c.metadata.get('source', 'unknown') for c in chunks]

# Embed all chunks (may take 10-30s depending on corpus size)
vectors = embeddings_model.embed_documents(texts)
vectors = np.array(vectors)
print(f'Embedding matrix: {vectors.shape}')

## 2. UMAP Dimensionality Reduction

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
coords_2d = reducer.fit_transform(vectors)

embed_df = pd.DataFrame({
    'x': coords_2d[:, 0],
    'y': coords_2d[:, 1],
    'source': sources,
    'text_preview': [t[:80] for t in texts],
})
embed_df.head()

In [ ]:
fig = px.scatter(
    embed_df, x='x', y='y', color='source',
    hover_data=['text_preview'],
    title='FMCG Corpus Embedding Space (UMAP)',
    width=900, height=600
)
fig.show()

## 3. Intra-Document vs. Inter-Document Distances

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

unique_sources = embed_df['source'].unique()
intra_dists, inter_dists = [], []

for src in unique_sources:
    mask = embed_df['source'] == src
    src_vecs = vectors[mask]
    other_vecs = vectors[~mask]
    if len(src_vecs) > 1:
        d = cosine_distances(src_vecs)
        intra_dists.extend(d[np.triu_indices(len(d), k=1)].tolist())
    if len(src_vecs) > 0 and len(other_vecs) > 0:
        sample_other = other_vecs[np.random.choice(len(other_vecs), min(50, len(other_vecs)), replace=False)]
        inter_dists.extend(cosine_distances(src_vecs[:20], sample_other).flatten().tolist())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(intra_dists, bins=40, alpha=0.6, label='Same document', density=True)
ax.hist(inter_dists, bins=40, alpha=0.6, label='Different documents', density=True)
ax.set_xlabel('Cosine Distance')
ax.set_ylabel('Density')
ax.set_title('Intra- vs Inter-Document Chunk Distances')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean intra-doc distance:  {np.mean(intra_dists):.3f}')
print(f'Mean inter-doc distance:  {np.mean(inter_dists):.3f}')
print(f'Separation ratio: {np.mean(inter_dists)/np.mean(intra_dists):.2f}x')

## 4. Confusion Zones — Different Documents That Land Close Together

In [ ]:
# Find cross-document pairs with cosine distance < threshold
threshold = 0.15
confusion_pairs = []

for i in range(min(len(texts), 200)):
    for j in range(i+1, min(len(texts), 200)):
        if sources[i] != sources[j]:
            d = float(cosine_distances([vectors[i]], [vectors[j]])[0][0])
            if d < threshold:
                confusion_pairs.append({
                    'distance': round(d, 4),
                    'source_a': sources[i],
                    'text_a': texts[i][:120],
                    'source_b': sources[j],
                    'text_b': texts[j][:120],
                })

confusion_df = pd.DataFrame(confusion_pairs).sort_values('distance')
print(f'Found {len(confusion_df)} confusion pairs (distance < {threshold})')
confusion_df.head(10)

## 5. Save Outputs for Dashboard

In [ ]:
from analysis.utils import save_output

save_output(embed_df, 'embedding_projections.json')

distance_stats = {
    'mean_intra_doc': float(np.mean(intra_dists)),
    'mean_inter_doc': float(np.mean(inter_dists)),
    'std_intra_doc': float(np.std(intra_dists)),
    'std_inter_doc': float(np.std(inter_dists)),
    'separation_ratio': float(np.mean(inter_dists) / np.mean(intra_dists)),
    'n_confusion_pairs': len(confusion_df),
}
save_output(distance_stats, 'cluster_distances.json')
print('Saved.')